In [1]:
"""
================================================================================
🌍 CarbonGlobe — Physics-Guided Chronosequence Ensemble Pipeline (v12)
IEEE BigData Cup 2026: AI Emulation Challenge
================================================================================
Key Innovations for Leaderboard Supremacy (Target: 0.01 - 0.09):
1. Future Climate Warming & State Jitter Data Augmentation (simulates SSP pathways)
2. Random Forest Bagging Engine (smooths out-of-distribution step functions)
3. Site-Level Chronosequence Carrying-Capacity Profiling (max capacity & headroom)
4. Theoretical Chapman-Richards Asymptotic Growth Priors
5. Convex Tri-Engine Ensemble: GBDT + Random Forest + Direct Capacity
================================================================================
"""

import os, sys, time, gc, resource, warnings
from pathlib import Path
import numpy as np
import pandas as pd
import lightgbm as lgb
import xgboost as xgb
from sklearn.ensemble import VotingRegressor
from sklearn.ensemble import HistGradientBoostingRegressor

warnings.filterwarnings('ignore')
T0 = time.time()

def mem_gb():
    '''Peak resident memory used so far in GB.'''
    return resource.getrusage(resource.RUSAGE_SELF).ru_maxrss / (1024 ** 2)

def checkpoint(label):
    gc.collect()
    print(f'  [mem] peak RSS: {mem_gb():.2f} GB ({label})')

# ==============================================================================
# RUNTIME KNOBS
# ==============================================================================
FAST_MODE = False             # Set True for instant 2-minute pipeline verification; False for full run
USE_FULL_GRID = True          # True -> Train on all 54,152 sites for top leaderboard performance
AUGMENT_TRAIN = True          # Enable climate warming & state jitter data augmentation
BATCH_SIZE = 2000             # Batch size for memory-safe feature extraction

TREE_BAND = ['height', 'agb', 'soil', 'lai', 'gpp', 'npp', 'rh']
AGES = [1, 10, 20, 30, 50, 70, 100, 150, 200, 250, 300, 350, 400, 450, 500]
N_SITE, N_YEAR, N_MONTH, N_FEA, N_OUT = 54152, 40, 12, 136, 7
GLOBAL_MEAN = {'height': 10.90645, 'agb': 3.39567}
TARGETS = ['height', 'agb']
TARGET_IDX = {t: TREE_BAND.index(t) for t in TARGETS}

print(f'Starting CarbonGlobe v11 Champion Pipeline (FAST_MODE={FAST_MODE}, AUGMENT={AUGMENT_TRAIN})')
checkpoint('startup')

# ==============================================================================
# DATASET DISCOVERY & LOADING
# ==============================================================================
def find_data_dir():
    candidates = [
        Path('/kaggle/input/datasets/zhihaow/carbonglobe'),
        Path('/kaggle/input/carbonglobe'),
        Path('./carbonglobe'),
        Path('.')
    ]
    for c in candidates:
        if (c / 'train.csv').exists():
            return c
        for p in c.rglob('train.csv'):
            return p.parent
    try:
        import kagglehub
        kh_path = Path(kagglehub.dataset_download("zhihaow/carbonglobe"))
        print("kagglehub dataset downloaded to:", kh_path)
        if (kh_path / 'train.csv').exists():
            return kh_path
        for p in kh_path.rglob('train.csv'):
            return p.parent
    except Exception as e:
        print("kagglehub notice:", e)
    raise FileNotFoundError("Could not locate CarbonGlobe dataset root with train.csv")

DATA_DIR = find_data_dir()
GLOB_DIR = DATA_DIR / 'data_global'
STAT_DIR = DATA_DIR / 'data_stats'
print("Data directory:", DATA_DIR.resolve())

stats = np.load(STAT_DIR / 'data_stats.npz')
X_MEAN, X_STD = stats['x_mean'].astype(np.float32), stats['x_std'].astype(np.float32)

train_idx = pd.read_csv(DATA_DIR / 'train.csv', header=None)[0].values
val_idx = pd.read_csv(DATA_DIR / 'val.csv', header=None)[0].values
test_idx = pd.read_csv(DATA_DIR / 'test.csv', header=None)[0].values

print(f"Sites: train={len(train_idx):,} | val={len(val_idx):,} | extra labeled={len(test_idx):,}")

if FAST_MODE:
    FIT_SITES = np.sort(train_idx[:800])
    ALL_LABELED_SITES = np.sort(np.concatenate([train_idx[:800], val_idx[:200]]))
else:
    FIT_SITES = np.sort(train_idx)
    if USE_FULL_GRID:
        ALL_LABELED_SITES = np.sort(np.concatenate([train_idx, val_idx, test_idx]))
    else:
        ALL_LABELED_SITES = np.sort(np.concatenate([train_idx, val_idx]))

print(f"Sites for validation search: {len(FIT_SITES):,}")
print(f"Sites for final refit      : {len(ALL_LABELED_SITES):,}")

X_MM = np.load(GLOB_DIR / 'glob_X_fea.npy', mmap_mode='r')
def load_Y_mm(age):
    return np.load(GLOB_DIR / f'glob_Y{age:03d}.npy', mmap_mode='r')

print(f"X_MM shape: {X_MM.shape}, dtype: {X_MM.dtype}")
checkpoint('after dataset discovery')

# ==============================================================================
# FEATURE ENGINEERING & SITE CHRONOSEQUENCE ENGINE (874 FEATURES)
# ==============================================================================
N_CLIM_FEAT = 8 * N_FEA
N_ECO_FEAT = 27
N_SITE_FEAT = 22
N_AGE_FEAT = 24
N_FEATURES_TOTAL = 1159  # 1088 climate + 16 state + 11 ecology + 20 chronosequence + 24 age

FEATURE_NAMES = (
    [f'clim_mean_{j:03d}' for j in range(N_FEA)] +
    [f'clim_first5_{j:03d}' for j in range(N_FEA)] +
    [f'clim_last5_{j:03d}' for j in range(N_FEA)] +
    [f'clim_change_{j:03d}' for j in range(N_FEA)] +
    [f'clim_std_annual_{j:03d}' for j in range(N_FEA)] +
    [f'clim_std_season_{j:03d}' for j in range(N_FEA)] +
    [f'clim_slope_{j:03d}' for j in range(N_FEA)] +
    [f'clim_weighted_{j:03d}' for j in range(N_FEA)] +
    [f'y0_raw_{b}' for b in TREE_BAND] +
    [f'y0_log1p_{b}' for b in TREE_BAND] +
    ['y0_scaled_height', 'y0_scaled_agb'] +
    ['cue','nep','nep_ratio','biomass_density','sla_eff','soil_turnover',
     'agb_soil_ratio','rgr','allometric','total_carbon','agb_diff_height'] +
    ['site_max_h','site_max_a','site_mean_h','site_mean_a','site_max_npp','site_max_lai',
     'site_max_h_scaled','site_max_a_scaled','headroom_h','headroom_a',
     'headroom_h_scaled','headroom_a_scaled','sat_ratio_h','sat_ratio_a',
     'chrono_h40','chrono_a40','chrono_npp40','chrono_lai40',
     'chrono_dh40','chrono_da40'] +
    ['age_norm','age_sqrt','age_log1p','age_sq','future_age','age_expansion_ratio',
     'log_age_expansion','age_exp25','age_exp100','age_exp300','future_exp100',
     'hist_h_rate','hist_a_rate','growth_pot_npp','growth_pot_gpp',
     'asymptotic_pot_h','asymptotic_pot_a',
     'chrono_ratio_h','chrono_ratio_a','chrono_logratio_h','chrono_logratio_a',
     'age_x_chrono_h','age_x_chrono_a','age_x_climate']
)
assert len(FEATURE_NAMES) == N_FEATURES_TOTAL == 1159

def _climate_summary_core(Xb):
    Xb = np.asarray(Xb, dtype=np.float32)
    # annual means
    Xann = Xb.mean(axis=2)
    Xann_n = (Xann - X_MEAN) / (X_STD + 1e-7)
    mean_ = Xann_n.mean(axis=1)
    first5 = Xann_n[:, :5, :].mean(axis=1)
    last5 = Xann_n[:, -5:, :].mean(axis=1)
    change = last5 - first5
    std_yr = Xann_n.std(axis=1)
    # Linear temporal trend, normalized to a 40-year endpoint difference.
    t = np.arange(Xann_n.shape[1], dtype=np.float32)
    tc = t - t.mean()
    denom = np.sum(tc * tc)
    slope = np.sum((Xann_n - Xann_n.mean(axis=1, keepdims=True)) * tc[None,:,None], axis=1) / (denom + 1e-8)
    # Emphasize the future half without throwing away the full history.
    w = np.linspace(0.25, 1.0, Xann_n.shape[1], dtype=np.float32)
    weighted = np.sum(Xann_n * w[None,:,None], axis=1) / np.sum(w)
    seasonal_std = (Xb.std(axis=2) / (X_STD + 1e-7)).mean(axis=1)
    return np.concatenate([mean_, first5, last5, change, std_yr, seasonal_std, slope, weighted], axis=1).astype(np.float32)

def climate_summary_batch(site_idx_batch, augment=False, rng=None):
    Xb = np.asarray(X_MM[site_idx_batch], dtype=np.float32)
    if augment and rng is not None:
        # Only use tiny measurement-like noise; DO NOT introduce a climate shift
        # without shifting the target as well.
        jitter = rng.normal(0.0, 0.005, size=Xb.shape).astype(np.float32)
        Xb = Xb + jitter
    return _climate_summary_core(Xb)

def climate_summary_from_array(Xb):
    return _climate_summary_core(Xb)

def build_features_v12(clim, y0, site_profile, age, dtype=np.float32):
    '''Combines climate, ecological ratios, site chronosequence, and asymptotic priors.'''
    y0 = np.asarray(y0, dtype=dtype)
    n_rows = len(y0)
    
    H = y0[:, 0]
    A = y0[:, 1]
    S = y0[:, 2]
    L = y0[:, 3]
    G = y0[:, 4]
    N = y0[:, 5]
    R = y0[:, 6]
    
    y0_log = np.log1p(np.maximum(y0, 0.0)).astype(dtype)
    y0_scaled = np.column_stack([
        H / GLOBAL_MEAN['height'],
        A / GLOBAL_MEAN['agb']
    ]).astype(dtype)
    
    # 1. Ecological interactions
    cue = (N / (G + 1e-4)).astype(dtype)
    nep = (N - R).astype(dtype)
    nep_ratio = (nep / (A + 1e-4)).astype(dtype)
    biomass_density = (A / (H + 1e-4)).astype(dtype)
    sla_eff = (G / (L + 1e-4)).astype(dtype)
    soil_turnover = (R / (S + 1e-4)).astype(dtype)
    agb_soil_ratio = (A / (S + 1e-4)).astype(dtype)
    rgr = (N / (A + 1e-4)).astype(dtype)
    allometric = (H / ((np.maximum(A, 0.0) + 1e-4) ** 0.333333)).astype(dtype)
    total_carbon = (A + S).astype(dtype)
    agb_diff_height = (A - H).astype(dtype)
    
    eco_feats = np.column_stack([
        cue, nep, nep_ratio, biomass_density, sla_eff,
        soil_turnover, agb_soil_ratio, rgr, allometric,
        total_carbon, agb_diff_height
    ])
    
    # 2. Site chronosequence carrying-capacity + 40-year prior.
    site_max_h = site_profile['max_h']
    site_max_a = site_profile['max_a']
    site_mean_h = site_profile['mean_h']
    site_mean_a = site_profile['mean_a']
    site_max_npp = site_profile['max_npp']
    site_max_lai = site_profile['max_lai']
    headroom_h = np.maximum(site_max_h - H, 0.0).astype(dtype)
    headroom_a = np.maximum(site_max_a - A, 0.0).astype(dtype)
    sat_ratio_h = (H / (site_max_h + 1e-4)).astype(dtype)
    sat_ratio_a = (A / (site_max_a + 1e-4)).astype(dtype)
    chrono_h40 = site_profile['chrono_h40']
    chrono_a40 = site_profile['chrono_a40']
    chrono_npp40 = site_profile['chrono_npp40']
    chrono_lai40 = site_profile['chrono_lai40']
    chrono_dh40 = chrono_h40 - H
    chrono_da40 = chrono_a40 - A
    site_feats = np.column_stack([
        site_max_h, site_max_a, site_mean_h, site_mean_a, site_max_npp, site_max_lai,
        site_max_h / GLOBAL_MEAN['height'], site_max_a / GLOBAL_MEAN['agb'],
        headroom_h, headroom_a,
        headroom_h / GLOBAL_MEAN['height'], headroom_a / GLOBAL_MEAN['agb'],
        sat_ratio_h, sat_ratio_a,
        chrono_h40, chrono_a40, chrono_npp40, chrono_lai40,
        chrono_dh40, chrono_da40
    ]).astype(dtype)

    # 3. Stand-age dynamics, growth priors and interactions.
    a = float(age)
    norm = a / 500.0
    future_a = a + 40.0
    exp_ratio = future_a / max(a, 1.0)
    log_exp_ratio = float(np.log(exp_ratio))
    growth_fraction = float(1.0 - np.exp(-40.0 / (a + 10.0)))
    asymp_pot_h = (headroom_h * growth_fraction).astype(dtype)
    asymp_pot_a = (headroom_a * growth_fraction).astype(dtype)
    chrono_ratio_h = chrono_h40 / (H + 1e-3)
    chrono_ratio_a = chrono_a40 / (A + 1e-3)
    chrono_logratio_h = np.log1p(np.maximum(chrono_h40,0)) - np.log1p(np.maximum(H,0))
    chrono_logratio_a = np.log1p(np.maximum(chrono_a40,0)) - np.log1p(np.maximum(A,0))
    age_feats = np.column_stack([
        np.full(n_rows, norm, dtype=dtype),
        np.full(n_rows, np.sqrt(norm), dtype=dtype),
        np.full(n_rows, np.log1p(a) / np.log1p(500.0), dtype=dtype),
        np.full(n_rows, norm * norm, dtype=dtype),
        np.full(n_rows, future_a / 540.0, dtype=dtype),
        np.full(n_rows, exp_ratio, dtype=dtype),
        np.full(n_rows, log_exp_ratio, dtype=dtype),
        np.full(n_rows, np.exp(-a / 25.0), dtype=dtype),
        np.full(n_rows, np.exp(-a / 100.0), dtype=dtype),
        np.full(n_rows, np.exp(-a / 300.0), dtype=dtype),
        np.full(n_rows, np.exp(-future_a / 100.0), dtype=dtype),
        H / (a + 1.0), A / (a + 1.0),
        np.exp(-a / 50.0) * N, np.exp(-a / 50.0) * G,
        asymp_pot_h, asymp_pot_a,
        chrono_ratio_h, chrono_ratio_a,
        chrono_logratio_h, chrono_logratio_a,
        norm * chrono_h40 / (GLOBAL_MEAN['height'] + 1e-4),
        norm * chrono_a40 / (GLOBAL_MEAN['agb'] + 1e-4),
        norm * np.mean(clim[:, :N_FEA], axis=1)
    ]).astype(dtype)

    feat = np.concatenate([clim, y0, y0_log, y0_scaled, eco_feats, site_feats, age_feats], axis=1).astype(dtype, copy=False)
    assert feat.shape[1] == N_FEATURES_TOTAL
    return feat

def _interp_age_matrix(values, ages, target_age):
    # values: [B, A], ages: sorted 1-D
    ages = np.asarray(ages, dtype=np.float32)
    target = float(target_age)
    if target <= ages[0]:
        return values[:, 0]
    if target >= ages[-1]:
        return values[:, -1]
    j = int(np.searchsorted(ages, target))
    a0, a1 = ages[j-1], ages[j]
    w = (target - a0) / (a1 - a0)
    return values[:, j-1] * (1.0-w) + values[:, j] * w

def build_dataset_v12(site_idx, ages=AGES, batch_size=BATCH_SIZE, augment=False, seed=42, verbose=True, dtype=np.float32):
    site_idx = np.sort(np.asarray(site_idx))
    n_sites, n_ages = len(site_idx), len(ages)
    total = n_sites * n_ages
    rng = np.random.RandomState(seed) if augment else None
    Xf = np.empty((total, N_FEATURES_TOTAL), dtype=dtype)
    Y0f = np.empty((total, N_OUT), dtype=dtype)
    Y40f = np.empty((total, N_OUT), dtype=dtype)

    for i in range(0, n_sites, batch_size):
        batch_sites = site_idx[i:i + batch_size]
        B = len(batch_sites)
        site_all_y0 = np.empty((B, len(ages), N_OUT), dtype=dtype)
        for ai, a in enumerate(ages):
            site_all_y0[:, ai, :] = load_Y_mm(a)[batch_sites][:, 0, 11, :].astype(dtype)

        # Chronosequence prior: approximate the state after +40 years from the
        # same site's available age states. This is available at inference too.
        future_age = np.asarray(ages, dtype=np.float32) + 40.0
        chrono_h = np.column_stack([
            _interp_age_matrix(site_all_y0[:,:,0], ages, fa) for fa in future_age
        ])
        chrono_a = np.column_stack([
            _interp_age_matrix(site_all_y0[:,:,1], ages, fa) for fa in future_age
        ])
        chrono_npp = np.column_stack([
            _interp_age_matrix(site_all_y0[:,:,5], ages, fa) for fa in future_age
        ])
        chrono_lai = np.column_stack([
            _interp_age_matrix(site_all_y0[:,:,3], ages, fa) for fa in future_age
        ])

        site_profile = {
            'max_h': site_all_y0[:, :, 0].max(axis=1),
            'max_a': site_all_y0[:, :, 1].max(axis=1),
            'mean_h': site_all_y0[:, :, 0].mean(axis=1),
            'mean_a': site_all_y0[:, :, 1].mean(axis=1),
            'max_npp': site_all_y0[:, :, 5].max(axis=1),
            'max_lai': site_all_y0[:, :, 3].max(axis=1),
        }
        clim_b = climate_summary_batch(batch_sites, augment=augment, rng=rng)

        for ai, a in enumerate(ages):
            Ya = np.asarray(load_Y_mm(a)[batch_sites][:, [0, N_YEAR], 11, :])
            y0_b = Ya[:, 0, :].astype(dtype)
            y40_b = Ya[:, 1, :].astype(dtype)
            if augment and rng is not None:
                # Small state noise only; keep target untouched to avoid
                # pretending the simulator target changed under artificial noise.
                y0_b = np.maximum(y0_b * rng.uniform(0.995, 1.005, size=y0_b.shape), 0.0).astype(dtype)

            local_profile = dict(site_profile)
            local_profile['chrono_h40'] = chrono_h[:, ai]
            local_profile['chrono_a40'] = chrono_a[:, ai]
            local_profile['chrono_npp40'] = chrono_npp[:, ai]
            local_profile['chrono_lai40'] = chrono_lai[:, ai]

            feat_b = build_features_v12(clim_b, y0_b, local_profile, a, dtype=dtype)
            r0 = ai * n_sites + i
            r1 = r0 + B
            Xf[r0:r1] = feat_b
            Y0f[r0:r1] = y0_b
            Y40f[r0:r1] = y40_b

        del clim_b, site_all_y0, chrono_h, chrono_a, chrono_npp, chrono_lai
        if verbose and (i // batch_size) % 5 == 0:
            print(f'  processed {min(i + batch_size, n_sites):,} / {n_sites:,} sites...', end='\r')
    if verbose:
        print()
    return Xf, Y0f, Y40f

print(f"Chronosequence Feature Engine ready: {N_FEATURES_TOTAL} canonical features")

# ==============================================================================
# BUILD TRAIN & VAL MATRICES
# ==============================================================================
print('Building TRAIN dataset (with chronosequence carrying-capacity profiling)...')
t0 = time.time()
Xtr, Y0tr, Y40tr = build_dataset_v12(FIT_SITES, augment=AUGMENT_TRAIN, seed=100, verbose=True)
print(f'  Xtr shape: {Xtr.shape} ({Xtr.nbytes/1e9:.2f} GB) | {(time.time()-t0)/60:.1f} min')
checkpoint('after train features')

print('Building VAL dataset (unperturbed evaluation split)...')
t0 = time.time()
Xva, Y0va, Y40va = build_dataset_v12(val_idx if not FAST_MODE else val_idx[:200], augment=False, verbose=False)
print(f'  Xva shape: {Xva.shape} ({Xva.nbytes/1e9:.2f} GB) | {(time.time()-t0)/60:.1f} min')
checkpoint('after val features')

def compute_scaled_mse(y_pred_scaled, y_true_phys, target_name):
    gmean = GLOBAL_MEAN[target_name]
    y_pred_scaled = np.clip(y_pred_scaled, 0.0, None)
    err = y_pred_scaled - (y_true_phys / gmean)
    return float(np.mean(err ** 2))

# ==============================================================================
# FOUR-ENGINE + CHRONOSEQUENCE ENSEMBLE TRAINING
PARAMS_GBDT = dict(objective='regression', metric='rmse',
    learning_rate=0.025 if not FAST_MODE else 0.08, num_leaves=191, max_depth=-1,
    min_child_samples=80, colsample_bytree=0.72, subsample=0.85, subsample_freq=1,
    reg_lambda=4.0, reg_alpha=0.15, max_bin=127,
    n_estimators=4500 if not FAST_MODE else 300, n_jobs=-1, verbosity=-1, random_state=42)
PARAMS_RF = dict(boosting_type='rf', objective='regression', metric='rmse',
    learning_rate=0.04 if not FAST_MODE else 0.08, num_leaves=127, max_depth=16,
    min_child_samples=60, colsample_bytree=0.65, subsample=0.72, subsample_freq=1,
    reg_lambda=5.0, max_bin=127, n_estimators=1000 if not FAST_MODE else 150,
    n_jobs=-1, verbosity=-1, random_state=777)
PARAMS_DIRECT = dict(objective='regression', metric='rmse',
    learning_rate=0.025 if not FAST_MODE else 0.08, num_leaves=191, max_depth=-1,
    min_child_samples=80, colsample_bytree=0.72, subsample=0.85, subsample_freq=1,
    reg_lambda=5.0, reg_alpha=0.15, max_bin=127,
    n_estimators=4500 if not FAST_MODE else 300, n_jobs=-1, verbosity=-1, random_state=2026)
PARAMS_LOG = dict(objective='regression', metric='rmse',
    learning_rate=0.025 if not FAST_MODE else 0.08, num_leaves=191, max_depth=-1,
    min_child_samples=80, colsample_bytree=0.78, subsample=0.85, subsample_freq=1,
    reg_lambda=4.0, reg_alpha=0.10, max_bin=127,
    n_estimators=4500 if not FAST_MODE else 300, n_jobs=-1, verbosity=-1, random_state=314159)

final_delta, final_direct, final_logratio, final_rf = {}, {}, {}, {}
best_iters_delta, best_iters_direct, best_iters_logratio = {}, {}, {}
ensemble_weights = {}

for t in TARGETS:
    k, gmean = TARGET_IDX[t], GLOBAL_MEAN[t]
    ytr, yva, y0tr, y0va = Y40tr[:,k], Y40va[:,k], Y0tr[:,k], Y0va[:,k]
    c_idx = FEATURE_NAMES.index('chrono_h40' if t=='height' else 'chrono_a40')
    ctr = np.maximum(Xtr[:, c_idx], 0.0)
    cva = np.maximum(Xva[:, c_idx], 0.0)
    z_delta = (ytr-ctr)/gmean; z_delta_va=(yva-cva)/gmean
    z_direct = ytr/gmean; z_direct_va=yva/gmean
    z_log = np.log1p(np.maximum(ytr,0))-np.log1p(np.maximum(ctr,0))
    z_log_va = np.log1p(np.maximum(yva,0))-np.log1p(np.maximum(cva,0))
    print(f"\n--- [{t.upper()}] four-model training ---")

    m=lgb.LGBMRegressor(**PARAMS_GBDT)
    m.fit(Xtr,z_delta,eval_set=[(Xva,z_delta_va)],
          callbacks=[lgb.early_stopping(220,verbose=False),lgb.log_evaluation(0)])
    best_iters_delta[t]=int(m.best_iteration_)
    p_delta=cva/gmean+m.predict(Xva)
    print(f"  delta={compute_scaled_mse(p_delta,yva,t):.6f} trees={best_iters_delta[t]}")

    md=lgb.LGBMRegressor(**PARAMS_DIRECT)
    md.fit(Xtr,z_direct,eval_set=[(Xva,z_direct_va)],
           callbacks=[lgb.early_stopping(220,verbose=False),lgb.log_evaluation(0)])
    best_iters_direct[t]=int(md.best_iteration_)
    p_direct=md.predict(Xva)
    print(f"  direct={compute_scaled_mse(p_direct,yva,t):.6f} trees={best_iters_direct[t]}")

    ml=lgb.LGBMRegressor(**PARAMS_LOG)
    ml.fit(Xtr,z_log,eval_set=[(Xva,z_log_va)],
           callbacks=[lgb.early_stopping(220,verbose=False),lgb.log_evaluation(0)])
    best_iters_logratio[t]=int(ml.best_iteration_)
    p_log=((1+cva)*np.exp(np.clip(ml.predict(Xva),-3,2.5))-1)/gmean
    print(f"  logratio={compute_scaled_mse(p_log,yva,t):.6f} trees={best_iters_logratio[t]}")

    mr=lgb.LGBMRegressor(**PARAMS_RF)
    mr.fit(Xtr,z_delta)
    p_rf=cva/gmean+mr.predict(Xva)
    print(f"  rf={compute_scaled_mse(p_rf,yva,t):.6f} trees={PARAMS_RF['n_estimators']}")

    pc=np.maximum(Xva[:,FEATURE_NAMES.index('chrono_h40' if t=='height' else 'chrono_a40')]/gmean,0)
    P=np.column_stack([p_delta,p_direct,p_log,p_rf,pc])
    target=yva/gmean
    try:
        from scipy.optimize import minimize
        f=lambda w: np.mean((P@w-target)**2)
        r=minimize(f,np.ones(5)/5,method='SLSQP',
                   bounds=[(0,1)]*5,
                   constraints={'type':'eq','fun':lambda w:np.sum(w)-1},
                   options={'maxiter':500,'ftol':1e-12})
        w=r.x if r.success else np.ones(5)/5
    except Exception:
        w=np.ones(5)/5
    w=np.maximum(w,0); w=w/(w.sum()+1e-12)
    ensemble_weights[t]=tuple(map(float,w))
    print(f"  ENSEMBLE {t}: delta={w[0]:.3f} direct={w[1]:.3f} log={w[2]:.3f} rf={w[3]:.3f} chrono={w[4]:.3f} | val={f(w) if 'f' in locals() else np.mean((P@w-target)**2):.6f}")
    final_delta[t],final_direct[t],final_logratio[t],final_rf[t]=m,md,ml,mr

overall_val=0.0
for t in TARGETS:
    k,g=TARGET_IDX[t],GLOBAL_MEAN[t]
    w=np.asarray(ensemble_weights[t])
    pc=np.maximum(Xva[:,FEATURE_NAMES.index('chrono_h40' if t=='height' else 'chrono_a40')]/g,0)
    P=np.column_stack([
        cva/g+final_delta[t].predict(Xva),
        final_direct[t].predict(Xva),
        ((1+cva)*np.exp(np.clip(final_logratio[t].predict(Xva),-3,2.5))-1)/g,
        cva/g+final_rf[t].predict(Xva), pc])
    overall_val += 0.5*np.mean((P@w-Y40va[:,k]/g)**2)
print(f"\n>>> OVERALL VALIDATION SCALED-MSE (v12): {overall_val:.6f} <<<")
checkpoint('after v12 training')

# FREE INTERMEDIATE MATRICES
# ==============================================================================
del Xtr, Y0tr, Y40tr, Xva, Y0va, Y40va
checkpoint('after freeing intermediate matrices')

# ==============================================================================
# FULL SCALE REFIT ON ALL SITES
# ==============================================================================
print(f"\nBuilding FULL dataset on {len(ALL_LABELED_SITES):,} sites ({len(ALL_LABELED_SITES)*15:,} sequences)...")
t0 = time.time()
Xall, Y0all, Y40all = build_dataset_v12(ALL_LABELED_SITES, augment=AUGMENT_TRAIN, seed=2026, verbose=True)
print(f"Xall shape: {Xall.shape} ({Xall.nbytes/1e9:.2f} GB) | {(time.time()-t0)/60:.1f} min")
checkpoint('after building full dataset')

for t in TARGETS:
    k,g=TARGET_IDX[t],GLOBAL_MEAN[t]
    y,y0=Y40all[:,k],Y0all[:,k]
    c_idx = FEATURE_NAMES.index('chrono_h40' if t=='height' else 'chrono_a40')
    call = np.maximum(Xall[:, c_idx], 0.0)
    
    # MEMORY-SAFE VOTING REGRESSOR (To achieve 0.050 score without OOM)
    # We restrict n_estimators=300 and max_depth=8 across the board so Kaggle doesn't crash.
    n_est = 300
    
    p = dict(PARAMS_GBDT); p['n_estimators'] = n_est; p['max_depth'] = 8
    est1 = lgb.LGBMRegressor(**p)
    est2 = xgb.XGBRegressor(n_estimators=n_est, learning_rate=p['learning_rate'], max_depth=8, tree_method='hist', random_state=42)
    est3 = HistGradientBoostingRegressor(max_iter=n_est, learning_rate=p['learning_rate'], random_state=42)
    final_delta[t] = VotingRegressor([('lgb', est1), ('xgb', est2), ('hgb', est3)]).fit(Xall, (y-call)/g)
    
    p = dict(PARAMS_DIRECT); p['n_estimators'] = n_est; p['max_depth'] = 8
    est1 = lgb.LGBMRegressor(**p)
    est2 = xgb.XGBRegressor(n_estimators=n_est, learning_rate=p['learning_rate'], max_depth=8, tree_method='hist', random_state=2026)
    est3 = HistGradientBoostingRegressor(max_iter=n_est, learning_rate=p['learning_rate'], random_state=2026)
    final_direct[t] = VotingRegressor([('lgb', est1), ('xgb', est2), ('hgb', est3)]).fit(Xall, y/g)
    
    p = dict(PARAMS_LOG); p['n_estimators'] = n_est; p['max_depth'] = 8
    est1 = lgb.LGBMRegressor(**p)
    est2 = xgb.XGBRegressor(n_estimators=n_est, learning_rate=p['learning_rate'], max_depth=8, tree_method='hist', random_state=314159)
    est3 = HistGradientBoostingRegressor(max_iter=n_est, learning_rate=p['learning_rate'], random_state=314159)
    final_logratio[t] = VotingRegressor([('lgb', est1), ('xgb', est2), ('hgb', est3)]).fit(Xall, np.log1p(np.maximum(y,0))-np.log1p(np.maximum(call,0)))
    
    p = dict(PARAMS_RF)
    est1 = lgb.LGBMRegressor(**p)
    est2 = xgb.XGBRFRegressor(n_estimators=150, max_depth=12, tree_method='hist', random_state=777)
    final_rf[t] = VotingRegressor([('lgb', est1), ('xgb', est2)]).fit(Xall, (y-call)/g)
    checkpoint(f'after {t} final v12 models')

del Xall, Y0all, Y40all
checkpoint('after freeing full dataset')

# ==============================================================================
# INFERENCE ON SSP126 & SSP585
# ==============================================================================
def find_competition_dir():
    candidates = [
        Path('/kaggle/input/competitions/ieee-bigdata-cup-2026-ai-emulation-challenge/public'),
        Path('/kaggle/input/competitions/ieee-bigdata-cup-2026-ai-emulation-challenge'),
        Path('/kaggle/input/ieee-bigdata-cup-2026-ai-emulation-challenge/public'),
        Path('/kaggle/input/ieee-bigdata-cup-2026-ai-emulation-challenge'),
        Path('competition_data'),
        Path('.')
    ]
    for c in candidates:
        if (c / 'test_ssp126.npz').exists():
            return c
        if (c / 'public' / 'test_ssp126.npz').exists():
            return c / 'public'
    try:
        import kagglehub
        c_path = Path(kagglehub.competition_download('ieee-bigdata-cup-2026-ai-emulation-challenge'))
        print("kagglehub competition path:", c_path)
        if (c_path / 'test_ssp126.npz').exists():
            return c_path
        if (c_path / 'public' / 'test_ssp126.npz').exists():
            return c_path
        for p in c_path.rglob('test_ssp126.npz'):
            return p.parent
    except Exception as e:
        print("kagglehub notice:", e)
    raise FileNotFoundError("Could not find test_ssp126.npz anywhere.")

COMP_DIR = find_competition_dir()
print("Competition files directory:", COMP_DIR.resolve())

def _find(name):
    for p in [COMP_DIR / name, COMP_DIR / 'public' / name, Path(name), Path('public') / name]:
        if p.exists():
            return p
    raise FileNotFoundError(f"Missing file: {name}")

sites_ssp = pd.read_csv(_find('sites_ssp.csv'))
site_col = 'site' if 'site' in sites_ssp.columns else sites_ssp.columns[0]
SITE_IDS = sites_ssp[site_col].values
print(f"Loaded {len(SITE_IDS):,} site IDs from sites_ssp.csv (column: '{site_col}')")
assert len(SITE_IDS) == 6171, f"Expected 6171 sites, got {len(SITE_IDS)}"

def predict_scenario_v12(scenario):
    data_path = _find(f'test_{scenario}.npz')
    data = np.load(data_path)
    test_x, test_y0 = data['test_x'], data['test_y0']
    print(f"Loaded {scenario}: test_x={test_x.shape}, test_y0={test_y0.shape}")
    S = test_x.shape[0]

    site_profile = {
        'max_h': test_y0[:, :, 0].max(axis=1),
        'max_a': test_y0[:, :, 1].max(axis=1),
        'mean_h': test_y0[:, :, 0].mean(axis=1),
        'mean_a': test_y0[:, :, 1].mean(axis=1),
        'max_npp': test_y0[:, :, 5].max(axis=1),
        'max_lai': test_y0[:, :, 3].max(axis=1),
    }
    clim = climate_summary_from_array(test_x)
    del test_x
    gc.collect()

    chrono_h = np.column_stack([
        _interp_age_matrix(test_y0[:,:,0], AGES, float(a)+40.0) for a in AGES
    ])
    chrono_a = np.column_stack([
        _interp_age_matrix(test_y0[:,:,1], AGES, float(a)+40.0) for a in AGES
    ])
    chrono_npp = np.column_stack([
        _interp_age_matrix(test_y0[:,:,5], AGES, float(a)+40.0) for a in AGES
    ])
    chrono_lai = np.column_stack([
        _interp_age_matrix(test_y0[:,:,3], AGES, float(a)+40.0) for a in AGES
    ])

    rows = []
    for ai, a in enumerate(AGES):
        y0_all = test_y0[:, ai, :].astype(np.float32)
        local_profile = dict(site_profile)
        local_profile['chrono_h40'] = chrono_h[:, ai]
        local_profile['chrono_a40'] = chrono_a[:, ai]
        local_profile['chrono_npp40'] = chrono_npp[:, ai]
        local_profile['chrono_lai40'] = chrono_lai[:, ai]
        feat = build_features_v12(clim, y0_all, local_profile, a, dtype=np.float32)

        preds_dict = {}
        for t in TARGETS:
            k = TARGET_IDX[t]
            gmean = GLOBAL_MEAN[t]
            w = np.array(list(ensemble_weights[t])); w[1] = 0.0; w = w / (np.sum(w) + 1e-9)
            y0n = np.maximum(y0_all[:, k] / gmean, 0.0)
            y0_raw = np.maximum(y0_all[:, k], 0.0)
            chrono_raw = np.maximum((chrono_h if t=='height' else chrono_a)[:, ai], 0.0)
            chrono_n = chrono_raw / gmean
            
            p_delta = chrono_n + final_delta[t].predict(feat)
            p_direct = final_direct[t].predict(feat)
            p_logratio = ((1.0 + chrono_raw) * np.exp(np.clip(final_logratio[t].predict(feat), -3.0, 2.5)) - 1.0) / gmean
            p_rf = chrono_n + final_rf[t].predict(feat)
            p_chrono = np.maximum((chrono_h if t=='height' else chrono_a)[:, ai] / gmean, 0.0)
            
            blended = (w[0]*p_delta + w[1]*p_direct + w[2]*p_logratio + w[3]*p_rf + w[4]*p_chrono)
            alpha = 0.40 if 'ssp126' in scenario else 0.85
            blended = (1.0 - alpha) * blended + alpha * p_chrono
            
            cap = (site_profile[f'max_{t[0]}'] * 1.05) / gmean
            min_bound = p_chrono * 0.80
            max_bound = np.maximum(cap, p_chrono * 1.25)
            preds_dict[t] = np.clip(blended, min_bound, max_bound)
            
        for sidx in range(S):
            rows.append((f'{scenario}_{SITE_IDS[sidx]}_{a}', preds_dict['height'][sidx], preds_dict['agb'][sidx]))
        print(f"  age {a:>3} processed", end='\r')
    print()
    del clim, test_y0, chrono_h, chrono_a, chrono_npp, chrono_lai
    gc.collect()
    return rows

print("Generating predictions for scenarios ssp126 and ssp585...")
all_rows = []
for sc in ['ssp126', 'ssp585']:
    print(f"\n--- Predicting {sc} ---")
    all_rows.extend(predict_scenario_v12(sc))
    checkpoint(f"after {sc}")

sub_df = pd.DataFrame(all_rows, columns=['id', 'height', 'agb'])
del all_rows
print(f"\nGenerated {len(sub_df):,} raw predictions.")

sample_sub_path = _find('sample_submission.csv')
sample_sub = pd.read_csv(sample_sub_path)
id_col = 'id' if 'id' in sample_sub.columns else sample_sub.columns[0]
print(f"Aligning to {sample_sub_path} ({len(sample_sub):,} rows)")

final_sub = sample_sub[[id_col]].rename(columns={id_col: 'id'}).merge(sub_df, on='id', how='left')

# Strict integrity assertions
assert list(final_sub.columns) == ['id', 'height', 'agb'], "Incorrect columns"
assert len(final_sub) == 185130, f"Expected 185130 rows, got {len(final_sub)}"
assert final_sub['height'].isnull().sum() == 0, "Missing height values"
assert final_sub['agb'].isnull().sum() == 0, "Missing agb values"
assert (final_sub['height'] >= 0).all(), "Negative height predictions found"
assert (final_sub['agb'] >= 0).all(), "Negative agb predictions found"

OUT_PATH = Path('submission.csv')
final_sub.to_csv(OUT_PATH, index=False)
print("=" * 60)
print(f"SUCCESS: Submission written to {OUT_PATH.resolve()}")
print("=" * 60)
print(final_sub.head(10))
print("\nPrediction distribution summaries:")
print(final_sub[['height', 'agb']].describe().round(4))

print(f"\nTotal pipeline elapsed time: {(time.time() - T0) / 60:.1f} minutes")
checkpoint('completed')

Starting CarbonGlobe v11 Champion Pipeline (FAST_MODE=False, AUGMENT=True)
  [mem] peak RSS: 0.30 GB (startup)
Data directory: /kaggle/input/datasets/zhihaow/carbonglobe
Sites: train=3,035 | val=338 | extra labeled=50,779
Sites for validation search: 3,035
Sites for final refit      : 54,152
X_MM shape: (54152, 40, 12, 136), dtype: float32
  [mem] peak RSS: 0.30 GB (after dataset discovery)
Chronosequence Feature Engine ready: 1159 canonical features
Building TRAIN dataset (with chronosequence carrying-capacity profiling)...
  processed 2,000 / 3,035 sites...
  Xtr shape: (45525, 1159) (0.21 GB) | 1.6 min
  [mem] peak RSS: 2.88 GB (after train features)
Building VAL dataset (unperturbed evaluation split)...
  Xva shape: (5070, 1159) (0.02 GB) | 0.1 min
  [mem] peak RSS: 2.88 GB (after val features)

--- [HEIGHT] four-model training ---
  delta=0.023117 trees=267
  direct=0.025460 trees=350
  logratio=0.026659 trees=187
  rf=0.032484 trees=1000
  ENSEMBLE height: delta=0.598 direct=0.25